# Demo 10g — Use cases on real data

**Class 10 · G1** (slides: `slides/10_clustering.md`)

One method per task, with the check that tells whether it worked:

* **Customer-like segments** (wine, 6 497 samples, 11 measurements): $k$-means and GMM, BIC for $k$, soft memberships.
* **Topics** (BBC news, 1 000 articles): NMF topics against $k$-means on tf-idf.
* **Anomalies** (indoor climate, 3 290 hours): DBSCAN noise as the alarm.
* **Taxonomy** (digits): Ward tree of the ten class means.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from kneeliverse import kneedle
from scipy.cluster import hierarchy as hc
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.metrics import normalized_mutual_info_score as nmi
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from faa import clustering as cl
from faa import data, demo, plotlib
from faa.demo import show, slide

demo.setup("10")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
COLS = [BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY]
SEED = 0

## Slide: Segments in the wine data

6 497 wines, 11 physico-chemical features, standardised. No label is used. The first 1 599 rows of the file are the
red wines, the rest white; that column is kept aside to check what the segments found.

In [ ]:
Xw, qual, names = data.load_table("wine_quality", "quality")
Xw = StandardScaler().fit_transform(Xw)
red = np.r_[np.ones(1599), np.zeros(len(Xw) - 1599)].astype(int)
ks = range(1, 9)
bic = [GaussianMixture(k, covariance_type="full", n_init=2, random_state=SEED).fit(Xw).bic(Xw) for k in ks]
J = np.array([KMeans(k, n_init=5, random_state=SEED).fit(Xw).inertia_ for k in ks])
k_knee = int(list(ks)[kneedle.knee(np.c_[list(ks), J])])
sil = {
    k: silhouette_score(Xw, KMeans(k, n_init=5, random_state=SEED).fit_predict(Xw), sample_size=2000, random_state=SEED)
    for k in range(2, 7)
}
slide("Segments in the wine data")
print(
    f"rows 1-1599 treated as red: total sulfur dioxide mean red {Xw[red == 1, names.index('total_sulfur_dioxide')].mean():.2f}, white {Xw[red == 0, names.index('total_sulfur_dioxide')].mean():.2f} (z-scores)"
)
print(
    f"BIC picks k = {list(ks)[int(np.argmin(bic))]}; Kneedle on the inertia: k = {k_knee}; best silhouette k = {max(sil, key=sil.get)} ({max(sil.values()):.3f})"
)
km2 = KMeans(2, n_init=10, random_state=SEED).fit(Xw)
g2 = GaussianMixture(2, n_init=3, random_state=SEED).fit(Xw)
print(f"k = 2 against red/white: k-means ARI {ari(red, km2.labels_):.3f}, GMM ARI {ari(red, g2.predict(Xw)):.3f}")

## Slide: Soft segments: who is in between?

The mixture of $k = 2$ gives every wine a membership. The wines with high entropy sit between the two segments:
the ones a hard split would assign by a coin toss.

In [ ]:
U = g2.predict_proba(Xw)
ent = cl.entropy(U)
p = PCA(2).fit_transform(Xw)
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.6))
ax[0].scatter(*p.T, c=np.where(red == 1, RED, BLUE), s=2, lw=0, alpha=0.6)
ax[0].set_title("true colour (kept aside)"), ax[0].set_xlabel("PC1"), ax[0].set_ylabel("PC2")
sc = ax[1].scatter(*p.T, c=ent, s=2 + 6 * ent, cmap="viridis", lw=0)
f.colorbar(sc, ax=ax[1], label="entropy (bits)", shrink=0.8)
ax[1].set_title("GMM entropy"), ax[1].set_xlabel("PC1")
plotlib.save(f, "g1_wine")
slide("Soft segments")
amb = ent > 0.5
pred = g2.predict(Xw)
if (pred == red).mean() < 0.5:
    pred = 1 - pred  # the mixture numbers its components arbitrarily: align them with the colours
print(f"{amb.sum()} wines ({amb.mean():.3f}) have entropy above 0.5 bit")
print(
    f"agreement with red/white: all wines {(pred == red).mean():.3f}, the ambiguous wines {(pred[amb] == red[amb]).mean():.3f}, the certain wines {(pred[~amb] == red[~amb]).mean():.3f}"
)

## Slide: Topics in news: NMF against $k$-means

tf-idf matrix of 1 000 BBC articles (five categories, never shown to the models). NMF gives each article a weight
per topic (soft); $k$-means on the same matrix gives one label. NMI and ARI against the categories.

In [ ]:
texts, cat = data.load_news()
vec = TfidfVectorizer(max_features=2000, stop_words="english", min_df=3)
T = vec.fit_transform(texts)  # n x words, sparse
words = np.array(vec.get_feature_names_out())
Td = T.toarray()
_, Wt, Ht, _ = pynnmf.nmf_hals(Td.T, 5, n=200, seed=SEED, init="nndsvd")  # Wt: words x 5, Ht: 5 x n
Uraw = Ht.T  # the scale of W and H is free: a topic with a large W column gets a small H
Unm = cl.nmf_membership(Wt, Ht)  # the scale of each W column moved into H, then each row sums to 1
svd = TruncatedSVD(50, random_state=SEED).fit_transform(T)
svd /= np.linalg.norm(svd, axis=1, keepdims=True)
kmt = KMeans(5, n_init=10, random_state=SEED).fit(svd)
slide("Topics in news")
rows = [
    ("NMF argmax of raw H", ari(cat, Uraw.argmax(1)), nmi(cat, Uraw.argmax(1))),
    ("NMF, W columns rescaled", ari(cat, Unm.argmax(1)), nmi(cat, Unm.argmax(1))),
    ("k-means on SVD(50)", ari(cat, kmt.labels_), nmi(cat, kmt.labels_)),
]
show(pl.DataFrame(rows, schema=["method", "ARI", "NMI"], orient="row"))
for j in range(5):
    top = words[np.argsort(-Wt[:, j])[:6]]
    maj = pl.Series(cat[Unm.argmax(1) == j]).mode()[0]
    print(f"topic {j} ({maj}): {', '.join(top)}")
print(f"articles whose largest topic weight is below 0.5: {(Unm.max(1) < 0.5).sum()} of {len(Unm)}")

## Slide: Anomalies: noise points as an alarm

Sixteen hourly series of one house (8 temperatures, 8 humidities), 3 290 hours. DBSCAN has no "anomaly" class:
the points that belong to no dense region are the alarm. $\varepsilon$ from the knee of the $k$-distance graph.

In [ ]:
S, cols = data.load_indoor_climate()
Xi = StandardScaler().fit_transform(S.T)
m = 2 * Xi.shape[1]
d = np.sort(NearestNeighbors(n_neighbors=m).fit(Xi).kneighbors(Xi)[0][:, -1])
eps = float(d[kneedle.knee(np.c_[np.arange(len(d)), d])])
rows = []
for scale in (0.6, 0.8, 1.0, 1.25):
    lab = DBSCAN(eps=eps * scale, min_samples=m).fit_predict(Xi)
    rows.append((eps * scale, len(set(lab)) - (-1 in lab), int((lab == -1).sum()), float((lab == -1).mean())))
slide("Anomalies: noise points as an alarm")
print(f"hours {Xi.shape[0]}, features {Xi.shape[1]}, min_samples {m}, knee eps {eps:.2f}")
show(pl.DataFrame(rows, schema=["eps", "clusters", "noise hours", "share"], orient="row"))
lab = DBSCAN(eps=eps, min_samples=m).fit_predict(Xi)
z = np.abs(Xi[lab == -1]).mean(0) if (lab == -1).any() else np.zeros(Xi.shape[1])
print("features that deviate most in the flagged hours:", [cols[i] for i in np.argsort(-z)[:3]])
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.3))
ax.plot(Xi[:, cols.index("T1")], color=GREY, lw=0.5)
idx = np.flatnonzero(lab == -1)
ax.scatter(idx, Xi[idx, cols.index("T1")], color=RED, s=4, zorder=3)
ax.set_xlabel("hour"), ax.set_ylabel("T1 (z-score)")
plotlib.save(f, "g1_anomaly")

## Slide: A taxonomy with a tree: the ten digit classes

Average image of each digit, Ward linkage on the 64 pixels: the digits that look alike merge first. A tree answers
"which classes are siblings", a question a flat clustering cannot.

In [ ]:
Xg, yg, _ = data.load_table("digits", "target")
means = np.array([Xg[yg == c].mean(0) for c in range(10)])
Z = hc.linkage(means, "ward")
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.5))
hc.dendrogram(Z, labels=[str(c) for c in range(10)], ax=ax, color_threshold=0.6 * Z[-1, 2], leaf_font_size=7)
ax.grid(False), ax.set_ylabel("merge height")
plotlib.save(f, "g1_taxonomy")
slide("A taxonomy with a tree")
order = [int(x) for x in hc.leaves_list(Z)]
print(
    "leaf order:",
    order,
    "; first merges (class, class, height):",
    [(int(a), int(b), round(float(h), 1)) for a, b, h, _ in Z[:3]],
)
km = KMeans(10, n_init=10, random_state=SEED).fit(Xg)
gm = GaussianMixture(10, covariance_type="diag", n_init=2, random_state=SEED).fit(
    PCA(20, random_state=SEED).fit_transform(Xg)
)
print(
    f"digits, 10 clusters: k-means ARI {ari(yg, km.labels_):.3f}, GMM (PCA 20, diag) ARI {ari(yg, gm.predict(PCA(20, random_state=SEED).fit_transform(Xg))):.3f}"
)

## Figure: topics against categories

Mean NMF membership of each category (rows) in each topic (columns): the soft rows follow the editors' categories.

In [ ]:
cats = sorted(set(cat))
Mc = np.array([Unm[cat == c_].mean(0) for c_ in cats])
f, ax = plt.subplots(figsize=(plotlib.COL, 1.6))
im = ax.imshow(Mc, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(len(cats)), cats), ax.set_xticks(range(5), [f"topic {j}" for j in range(5)], fontsize=6)
ax.grid(False)
for i_ in range(len(cats)):
    for j_ in range(5):
        ax.text(
            j_, i_, f"{Mc[i_, j_]:.2f}", ha="center", va="center", fontsize=6, color="w" if Mc[i_, j_] < 0.5 else "k"
        )
plotlib.save(f, "g1_topics")
slide("Topics against categories")
print("largest mean membership per category:", {c_: round(float(Mc[i_].max()), 2) for i_, c_ in enumerate(cats)})

## Wine: six methods against the colour

Same wines, no label in the fit, $k = 2$ (3 000-wine sample for speed). The mixture and NMF are scored for several
$k$ too: more components than two split the colours further, so the ARI to red/white falls. NMF runs on the positive and
negative parts of the standardised wines and its rows are rescaled by $W$ (`cl.nmf_membership`).

In [ ]:
from sklearn.cluster import AgglomerativeClustering

S = np.random.default_rng(SEED).choice(len(Xw), 3000, replace=False)
Xs, rs = Xw[S], red[S]
lift = np.c_[np.maximum(Xs, 0), np.maximum(-Xs, 0)]


def nmf_labels(k):
    _, W_, H_, _ = pynnmf.nmf_hals(lift.T, k, n=300, seed=SEED, init="random")
    return cl.nmf_membership(W_, H_).argmax(1)


d_ = np.sort(NearestNeighbors(n_neighbors=22).fit(Xs).kneighbors(Xs)[0][:, -1])
eps_w = float(d_[kneedle.knee(np.c_[np.arange(len(d_)), d_])])
labs = {
    "k-means": KMeans(2, n_init=10, random_state=SEED).fit_predict(Xs),
    "Ward": AgglomerativeClustering(2, linkage="ward").fit_predict(Xs),
    "DBSCAN": DBSCAN(eps=eps_w, min_samples=22).fit_predict(Xs),
    "GMM": GaussianMixture(2, n_init=3, random_state=SEED).fit_predict(Xs),
    "FCM m=1.1": cl.fcm(Xs, 2, m=1.1, iters=500, seed=SEED)[1].argmax(1),
    "FCM m=2": cl.fcm(Xs, 2, m=2.0, iters=500, seed=SEED)[1].argmax(1),
    "NMF": nmf_labels(2),
}
rows = []
for name, lab in labs.items():
    ok = lab >= 0
    sil = silhouette_score(Xs[ok], lab[ok]) if len(set(lab[ok])) > 1 else float("nan")
    rows.append((name, ari(rs, lab), sil, int((lab < 0).sum())))
slide("Wine: six methods against the colour")
show(pl.DataFrame(rows, schema=["method", "ARI to red/white", "silhouette", "noise"], orient="row"))
print(f"DBSCAN eps from the knee: {eps_w:.2f}")

rows = []
for k in (2, 3, 4, 6, 8):
    g = GaussianMixture(k, n_init=3, random_state=SEED).fit(Xs)
    rows.append(
        (
            k,
            ari(rs, g.predict(Xs)),
            g.bic(Xs),
            ari(rs, nmf_labels(k)),
            ari(rs, KMeans(k, n_init=10, random_state=SEED).fit_predict(Xs)),
        )
    )
slide("Wine: components against the colour")
show(pl.DataFrame(rows, schema=["k", "ARI GMM", "BIC GMM", "ARI NMF", "ARI k-means"], orient="row"))

f, ax = plt.subplots(figsize=(plotlib.FULL, 1.4))
names = list(labs)
vals = [ari(rs, labs[n]) for n in names]
ax.bar(names, vals, color=[BLUE if v > 0.5 else GREY for v in vals])
ax.set_ylabel("ARI to red/white")
ax.tick_params(axis="x", labelsize=6)
plotlib.save(f, "g1_wine_methods")